# FractalTrainer — Sprint 19 scale stress test (N=1000 experts)

Addresses paper §5 limitation: *"Scale untested. The registry tops out at N=15 in the main experiments [...] Behavior at N=1000+ (realistic deployment scale) is not validated."*

### What this notebook measures

Builds **250 subsets × 4 seeds = 1000 registry experts** plus **250 held-out experts** (5th seed per subset) for routing tests. At N ∈ {12, 50, 100, 250, 500, 1000} subsample the registry **by subset** (keep all 4 seeds per chosen subset) so within-task pairs exist at every N, then report:

1. **Routing accuracy (top-1, top-3)** — held-out signature lookup against registry, check if nearest entry is same-subset.
2. **Routing latency** — per-query wall-time of linear-scan nearest-neighbor.
3. **Within-task vs cross-task distance gap** — does the paper's N=15 clean gap (within 3.0 / cross 10.7) survive?
4. **Verdict distribution** at paper's fixed MNIST-MLP thresholds (match=5.0, spawn=7.0).

### Pre-registered pass gates at N=1000

- Routing accuracy ≥ 90%
- Per-query p95 latency < 50 ms
- Within / cross distance gap > 0

### Cost & runtime

| Mode | N registry | Held-outs | Total trainings | T4 GPU | CPU |
|---|---|---|---|---|---|
| `SMOKE=True` | 40 | 10 | 50 | ~2 min | ~8 min |
| `SMOKE=False` | 1000 | 250 | 1250 | ~60 min | ~3 hr |

**Free Colab is risky** — 90 min inactivity timeout may kill the run. Keep the tab focused, or use Colab Pro ($10/mo) for a safer session. The SMOKE path is free insurance: validates the whole pipeline before paying for the full run.

### Results persistence

Results save to three places (belt & braces):
1. `/content/` — ephemeral, lost on disconnect
2. Google Drive (if mounted) — persists
3. Printed at the end of this notebook — persists in Colab's saved notebook state

**Prerequisites** — latest FractalTrainer master must be pushed to GitHub (`git push origin master`) before the clone cell runs, otherwise this notebook's dependencies won't be there.

## Cell 1. Config — EDIT HERE before running

In [ ]:
# Config toggles — smoke first, full run second.
SMOKE = True             # True = 40+10 experts, ~2 min. False = 1000+250, ~60 min.
USE_DRIVE = True         # mount Google Drive for persistent result saves
DRIVE_SAVE_DIR = "FractalTrainer_results"  # folder name inside /content/drive/MyDrive/
N_STEPS_PER_EXPERT = 200  # per-expert training steps (paper default was 500 for seeds)

if SMOKE:
    N_SUBSETS = 10
    SEEDS_PER_SUBSET = 4
    N_GRID = [8, 16, 32, 40]
else:
    N_SUBSETS = 250
    SEEDS_PER_SUBSET = 4
    N_GRID = [12, 50, 100, 250, 500, 1000]

print(f"mode={'SMOKE' if SMOKE else 'FULL'}  "
      f"subsets={N_SUBSETS}  seeds/subset={SEEDS_PER_SUBSET}  "
      f"registry_max={N_SUBSETS * SEEDS_PER_SUBSET}  held_outs={N_SUBSETS}")

## Cell 2. Clone repo + device check

In [ ]:
import os, sys, subprocess, time

REPO_URL = "https://github.com/vegarjr/FractalTrainer.git"
REPO_DIR = "/content/FractalTrainer"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
head = subprocess.check_output(["git", "-C", REPO_DIR, "log", "--oneline", "-1"]).decode().strip()
print("repo at", REPO_DIR, "\nHEAD:", head)

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__}  device={DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  CPU mode — set Runtime → Change runtime type → GPU (T4) for 5-10x speedup.")

In [ ]:
# Mount Drive for persistent result saves (so runtime disconnect doesn't lose everything)
DRIVE_PATH = None
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_PATH = os.path.join("/content/drive/MyDrive", DRIVE_SAVE_DIR)
        os.makedirs(DRIVE_PATH, exist_ok=True)
        print("Drive mounted, will save to:", DRIVE_PATH)
    except Exception as e:
        print("Drive not available (ok if running locally):", e)
        DRIVE_PATH = None

## Cell 3. Load MNIST + pick fixed probe batch

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
train_ds = datasets.MNIST("/content/mnist_data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST("/content/mnist_data", train=False, download=True, transform=transform)

# Fixed 100-image probe — identical across all experts (pinned once to device)
probe_rng = np.random.default_rng(0)
probe_idx = probe_rng.choice(len(test_ds), size=100, replace=False)
probe_X = torch.stack([test_ds[i][0] for i in probe_idx]).view(100, -1).to(DEVICE)
print(f"probe shape: {probe_X.shape}  device: {probe_X.device}")

# Pre-compute label index arrays once — avoids O(N) scans per expert
train_labels_np = np.array(train_ds.targets)

## Cell 4. Task catalogue — subset × seed grid

In [ ]:
from itertools import combinations

all_subsets = []
for k in (3, 4, 5, 6, 7):
    for combo in combinations(range(10), k):
        all_subsets.append(combo)

rng = np.random.default_rng(42)
perm = rng.permutation(len(all_subsets))
chosen = [all_subsets[i] for i in perm[:N_SUBSETS]]
REGISTRY_SEEDS = list(range(100, 100 + SEEDS_PER_SUBSET))   # e.g. [100,101,102,103]
HELDOUT_SEED = 9001

REGISTRY_TASKS = [(s, seed) for s in chosen for seed in REGISTRY_SEEDS]
print(f"distinct subsets: {len(chosen)} (out of {len(all_subsets)} possible k∈3..7)")
print(f"registry tasks:   {len(REGISTRY_TASKS)}  =  {len(chosen)} × {SEEDS_PER_SUBSET} seeds")
print(f"held-out tasks:   {len(chosen)}  (seed {HELDOUT_SEED} per subset)")
print(f"total trainings:  {len(REGISTRY_TASKS) + len(chosen)}")

## Cell 5. Training helper (shared by registry + held-outs)

In [ ]:
from fractaltrainer.integration.context_mlp import ContextAwareMLP
from fractaltrainer.integration.signatures import softmax_signature
from fractaltrainer.registry import FractalEntry, FractalRegistry
import torch.nn.functional as F


class _FlatSubset(Dataset):
    """MNIST filtered to a label subset, images flattened to 784-d."""
    def __init__(self, base, subset, label_array):
        self.base = base
        self.idx = np.where(np.isin(label_array, list(subset)))[0]
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        x, y = self.base[int(self.idx[i])]
        return x.view(-1), int(y)


def train_one_expert(subset, seed, n_steps=N_STEPS_PER_EXPERT, lr=0.01, batch_size=64):
    torch.manual_seed(seed); np.random.seed(seed)
    ds = _FlatSubset(train_ds, subset, train_labels_np)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True,
                        num_workers=0, drop_last=False)
    model = ContextAwareMLP(context_scale=0.0).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    step = 0; iterator = iter(loader)
    while step < n_steps:
        try:
            x, y = next(iterator)
        except StopIteration:
            iterator = iter(loader); x, y = next(iterator)
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        opt.zero_grad()
        loss = F.cross_entropy(model(x, context=None), y)
        loss.backward()
        opt.step()
        step += 1
    model.eval()
    sig = softmax_signature(model, probe_X)  # probe already on DEVICE
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return sig


# One-expert smoke test (skip if already proven)
_t = time.time()
_sig = train_one_expert((0, 1, 2), 42, n_steps=50)
print(f"smoke train ok  sig_l2={np.linalg.norm(_sig):.3f}  [{time.time()-_t:.1f}s for 50 steps]")

## Cell 6. Train all registry experts

In [ ]:
registry_full = FractalRegistry()
subset_to_entries = {}   # subset tuple → list of entry names, useful for subsampling

t0 = time.time()
for i, (subset, seed) in enumerate(REGISTRY_TASKS):
    sig = train_one_expert(subset, seed)
    task_name = "_".join(str(d) for d in subset)
    entry_name = f"{task_name}__s{seed}"
    entry = FractalEntry(
        name=entry_name,
        signature=sig,
        metadata={"task": task_name, "subset": list(subset), "seed": seed},
    )
    registry_full.add(entry)
    subset_to_entries.setdefault(tuple(subset), []).append(entry_name)
    if (i + 1) % max(1, len(REGISTRY_TASKS) // 20) == 0:
        dt = time.time() - t0
        eta = dt / (i + 1) * (len(REGISTRY_TASKS) - i - 1)
        print(f"  {i+1:>5d}/{len(REGISTRY_TASKS)}  elapsed={dt:>5.0f}s  eta={eta:>5.0f}s")

print(f"\nRegistry built: {len(registry_full)} experts in {time.time()-t0:.0f}s")
print(f"Subsets with full seed coverage: "
      f"{sum(1 for v in subset_to_entries.values() if len(v) == SEEDS_PER_SUBSET)}")

## Cell 7. Train held-out experts (one per subset, seed 9001)

In [ ]:
subset_to_heldout_sig = {}
t0 = time.time()
for i, subset in enumerate(chosen):
    sig = train_one_expert(subset, HELDOUT_SEED)
    subset_to_heldout_sig[tuple(subset)] = sig
    if (i + 1) % max(1, len(chosen) // 10) == 0:
        print(f"  held-out {i+1}/{len(chosen)}  elapsed={time.time()-t0:.0f}s")
print(f"\nHeld-outs done in {time.time()-t0:.0f}s.")

## Cell 8. Subset-preserving subsampler

Picks M subsets (with all seeds included) to give exactly `M × SEEDS_PER_SUBSET` entries, so within-task pairs always exist at every N.

In [ ]:
def subsample_by_subset(subset_to_entries, registry_full, N, seed=0):
    """Return a FractalRegistry with ~N entries, picking M=N/seeds_per_subset
    subsets and keeping ALL seeds for each — guarantees within-task pairs."""
    rng = np.random.default_rng(seed)
    M = max(1, N // SEEDS_PER_SUBSET)
    all_subsets_list = list(subset_to_entries.keys())
    pick_idx = rng.permutation(len(all_subsets_list))[:M]
    sub = FractalRegistry()
    for i in pick_idx:
        s = all_subsets_list[i]
        for name in subset_to_entries[s]:
            entry = registry_full.get(name)
            if entry is not None:
                sub.add(entry)
    return sub, [all_subsets_list[i] for i in pick_idx]


def within_cross_distances(reg):
    ents = reg.entries()
    sigs = np.stack([e.signature for e in ents])
    tasks = [tuple(e.metadata["subset"]) for e in ents]
    n = len(ents)
    D = np.linalg.norm(sigs[:, None, :] - sigs[None, :, :], axis=-1)
    within, cross = [], []
    for i in range(n):
        for j in range(i + 1, n):
            (within if tasks[i] == tasks[j] else cross).append(D[i, j])
    return np.array(within), np.array(cross)

## Cell 9. Distance + routing + verdict stats at each N

Uses paper's **fixed MNIST-MLP thresholds**: match_t=5.0, spawn_t=7.0 (paper §2). Skips empirical re-calibration at small N where it's unreliable.

In [ ]:
MATCH_T = 5.0    # from paper §2 — 95th percentile of within-task on seed registry
SPAWN_T = 7.0    # from paper §2 — 5th percentile of cross-task on seed registry

distance_stats, routing_stats, verdict_stats = {}, {}, {}

for N in N_GRID:
    reg, picked_subsets = subsample_by_subset(subset_to_entries, registry_full, N, seed=0)
    # --- distances ---
    w, c = within_cross_distances(reg)
    distance_stats[N] = {
        "n_entries": len(reg),
        "n_within_pairs": int(w.size), "n_cross_pairs": int(c.size),
        "within_mean": float(w.mean()) if w.size else None,
        "within_std":  float(w.std())  if w.size else None,
        "cross_mean":  float(c.mean()) if c.size else None,
        "cross_std":   float(c.std())  if c.size else None,
        "gap": float((c.mean() if c.size else 0.0) - (w.mean() if w.size else 0.0)),
    }
    # --- routing test with held-outs ---
    queries = [(s, subset_to_heldout_sig[tuple(s)]) for s in picked_subsets
               if tuple(s) in subset_to_heldout_sig]
    top1, top3, latencies = 0, 0, []
    for subset, qsig in queries:
        t = time.time()
        res = reg.find_nearest(qsig, k=3, query_name=str(subset))
        latencies.append((time.time() - t) * 1000.0)
        hit = [tuple(e.metadata["subset"]) for e in res.entries]
        if hit and hit[0] == tuple(subset): top1 += 1
        if tuple(subset) in hit: top3 += 1
    routing_stats[N] = {
        "n_queries": len(queries),
        "top1_acc": top1 / max(1, len(queries)),
        "top3_acc": top3 / max(1, len(queries)),
        "latency_ms_mean": float(np.mean(latencies)) if latencies else None,
        "latency_ms_p95":  float(np.percentile(latencies, 95)) if latencies else None,
    }
    # --- verdict distribution (in-subset held-outs only) ---
    v = {"match": 0, "compose": 0, "spawn": 0, "empty": 0}
    for subset, qsig in queries:
        d = reg.decide(qsig, MATCH_T, SPAWN_T)
        v[d.verdict] += 1
    tot = max(1, sum(v.values()))
    verdict_stats[N] = {"match_t": MATCH_T, "spawn_t": SPAWN_T,
                        "counts": v, "fractions": {k: n/tot for k, n in v.items()}}
    print(f"N={N:>5d}  entries={len(reg):>4d}  "
          f"within_pairs={int(w.size):>4d} cross_pairs={int(c.size):>5d}  "
          f"top1={routing_stats[N]['top1_acc']:.3f}  "
          f"top3={routing_stats[N]['top3_acc']:.3f}  "
          f"p95_lat={routing_stats[N]['latency_ms_p95']:.2f}ms  "
          f"gap={distance_stats[N]['gap']:+.3f}")

## Cell 10. Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
Ns = np.array([distance_stats[N]["n_entries"] for N in N_GRID])

ax = axes[0, 0]
ax.plot(Ns, [routing_stats[N]["top1_acc"] for N in N_GRID], "o-", label="top-1")
ax.plot(Ns, [routing_stats[N]["top3_acc"] for N in N_GRID], "s--", label="top-3")
ax.axhline(0.9, color="gray", ls=":", alpha=0.5, label="pass gate (0.9)")
ax.set_xscale("log"); ax.set_xlabel("Registry size N (actual)")
ax.set_ylabel("Routing accuracy"); ax.set_ylim(0, 1.02)
ax.set_title("Routing accuracy vs N"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(Ns, [routing_stats[N]["latency_ms_mean"] for N in N_GRID], "o-", label="mean")
ax.plot(Ns, [routing_stats[N]["latency_ms_p95"]  for N in N_GRID], "s--", label="p95")
ax.axhline(50.0, color="gray", ls=":", alpha=0.5, label="gate (50 ms)")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Registry size N")
ax.set_ylabel("Query latency (ms)"); ax.set_title("Linear-scan latency vs N"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1, 0]
w_means = [distance_stats[N]["within_mean"] for N in N_GRID]
c_means = [distance_stats[N]["cross_mean"]  for N in N_GRID]
w_stds  = [distance_stats[N]["within_std"]  for N in N_GRID]
c_stds  = [distance_stats[N]["cross_std"]   for N in N_GRID]
ax.errorbar(Ns, c_means, yerr=c_stds, fmt="s-", label="cross-task", capsize=3)
ax.errorbar(Ns, w_means, yerr=w_stds, fmt="o-", label="within-task", capsize=3)
ax.axhline(MATCH_T, color="gray", ls=":", alpha=0.6, label=f"match_t={MATCH_T}")
ax.axhline(SPAWN_T, color="darkorange", ls=":", alpha=0.6, label=f"spawn_t={SPAWN_T}")
ax.set_xscale("log"); ax.set_xlabel("Registry size N")
ax.set_ylabel("L2 signature distance"); ax.set_title("Within-task vs cross-task distance"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1, 1]
for verdict in ["match", "compose", "spawn"]:
    fracs = [verdict_stats[N]["fractions"][verdict] for N in N_GRID]
    ax.plot(Ns, fracs, "o-", label=f"in-subset {verdict}")
ax.set_xscale("log"); ax.set_xlabel("Registry size N"); ax.set_ylabel("Fraction")
ax.set_ylim(0, 1.02); ax.set_title(f"Verdict dist (fixed thresh {MATCH_T}/{SPAWN_T})")
ax.legend(); ax.grid(alpha=0.3)

fig.tight_layout()
_stem = "sprint19_scale_n1000" + ("_smoke" if SMOKE else "")
plt_path = f"/content/{_stem}.png"
fig.savefig(plt_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"plot saved: {plt_path}")

## Cell 11. Save results (3 destinations for redundancy)

In [ ]:
import json, shutil

payload = {
    "config": {
        "mode": "SMOKE" if SMOKE else "FULL",
        "n_subsets": N_SUBSETS, "seeds_per_subset": SEEDS_PER_SUBSET,
        "n_steps_per_expert": N_STEPS_PER_EXPERT,
        "registry_size_max": N_SUBSETS * SEEDS_PER_SUBSET,
        "n_heldouts": N_SUBSETS,
        "signature_dim": 1000,
        "match_threshold": MATCH_T, "spawn_threshold": SPAWN_T,
        "device": str(DEVICE),
        "git_head": head,
    },
    "distance_stats": distance_stats,
    "routing_stats":  routing_stats,
    "verdict_stats":  verdict_stats,
}
json_path = f"/content/{_stem}.json"
with open(json_path, "w") as fh:
    json.dump(payload, fh, indent=2, default=float)
print(f"1. Local: {json_path} / {plt_path}")

if DRIVE_PATH is not None:
    shutil.copy(json_path, DRIVE_PATH); shutil.copy(plt_path, DRIVE_PATH)
    print(f"2. Drive: {DRIVE_PATH}/")
else:
    print("2. Drive: skipped (USE_DRIVE=False or mount failed)")

print("3. Inline (below) — survives notebook save even if runtime dies.")

## Cell 12. Final report (captured in notebook output)

Prints every number. If you save the notebook (`File → Save`) after this cell runs, the output persists in the `.ipynb` even if the runtime disconnects.

In [ ]:
print("=" * 60)
print(f"SPRINT 19 SCALE TEST — {'SMOKE' if SMOKE else 'FULL'} MODE")
print("=" * 60)
print(f"git HEAD:       {head}")
print(f"device:         {DEVICE}")
print(f"subsets:        {N_SUBSETS} × {SEEDS_PER_SUBSET} seeds = {N_SUBSETS*SEEDS_PER_SUBSET} experts")
print(f"held-outs:      {N_SUBSETS} (seed {HELDOUT_SEED})")
print(f"thresholds:     match={MATCH_T}  spawn={SPAWN_T}  (paper §2 fixed)")
print()
print(f"{'N':>5s}  {'entries':>7s}  {'within':>8s}±std  {'cross':>8s}±std  {'gap':>6s}  "
      f"{'top-1':>5s}  {'top-3':>5s}  {'p95_ms':>7s}")
print("-" * 80)
for N in N_GRID:
    d = distance_stats[N]; r = routing_stats[N]
    w = f"{d['within_mean']:.2f}" if d['within_mean'] else "n/a"
    ws = f"{d['within_std']:.2f}"  if d['within_std']  else "n/a"
    c = f"{d['cross_mean']:.2f}"  if d['cross_mean']  else "n/a"
    cs = f"{d['cross_std']:.2f}"   if d['cross_std']   else "n/a"
    print(f"{N:>5d}  {d['n_entries']:>7d}  {w:>5s}±{ws:>4s}  {c:>5s}±{cs:>4s}  "
          f"{d['gap']:>+6.2f}  {r['top1_acc']:>.3f}  {r['top3_acc']:>.3f}  "
          f"{r['latency_ms_p95']:>7.2f}")

print()
print("=== Pre-registered pass gates (N = max) ===")
N_max = N_GRID[-1]
t1 = routing_stats[N_max]["top1_acc"]
lt = routing_stats[N_max]["latency_ms_p95"]
gp = distance_stats[N_max]["gap"]
print(f"  top-1 ≥ 0.9:   {t1:.3f}   {'✓ PASS' if t1 >= 0.9 else '✗ FAIL'}")
print(f"  p95 < 50 ms:   {lt:.2f}   {'✓ PASS' if lt < 50.0 else '✗ FAIL'}")
print(f"  gap > 0:       {gp:+.2f}  {'✓ PASS' if gp > 0 else '✗ FAIL'}")
print()
print("Copy-paste the JSON payload below into Reviews/46_v3_sprint19_scale.md (or similar)\n"
      "if Drive/download channels also fail:\n")
print(json.dumps(payload, indent=2, default=float))

## Cell 13. Trigger browser download (optional third backup)

In [ ]:
try:
    from google.colab import files
    files.download(json_path)
    files.download(plt_path)
    print("Downloads triggered in browser.")
except Exception as e:
    print("Colab files.download unavailable:", e)
    print(f"Files at: {json_path} / {plt_path}")

---

## What to do after Colab finishes

1. **Save the notebook** — `File → Save` or `Ctrl+S`. This preserves all cell outputs (including the final inline JSON) in your Colab-stored notebook. Even if runtime disconnects tomorrow, the outputs persist.
2. **Option A, Drive path:** if `USE_DRIVE=True`, results are already in `/content/drive/MyDrive/FractalTrainer_results/`. Open Google Drive in browser → download.
3. **Option B, browser download:** Cell 13 triggers `.json` + `.png` downloads to your machine.
4. **Option C, inline JSON:** the last print in Cell 12 spits the full JSON into the notebook output — copy-paste as a last resort.
5. **Back in the local repo:**
   ```
   cp ~/Downloads/sprint19_scale_n1000.{json,png} /home/vegar/Documents/FractalTrainer/results/
   # then commit
   ```
6. **Interpretation:** if top-1 ≥ 0.9 at N=1000 and the gap is still positive, the registry scales cleanly — paper §5 "scale untested" becomes "validated to N=1000." If top-1 drops below 0.9, the finding is that routing quality degrades with N — also publishable, and motivates switching to an ANN index (FAISS) instead of linear scan.
